# Garbage Classification (CIFAR-10)

**Image Classification** — Classify images into 10 categories.

Models: Simple CNN · ConvNeXt V2 Atto (via timm)  
Dataset: CIFAR-10 (60K images, 10 classes)  
Framework: PyTorch

## 2 · Project Overview

We classify CIFAR-10 images (32×32 RGB) into 10 categories: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.

This project demonstrates building a simple CNN from scratch and comparing it with a modern pretrained backbone (ConvNeXt V2 Atto).

## 3 · Learning Objectives

1. Build a CNN image classifier from scratch.
2. Use transfer learning with ConvNeXt V2 Atto.
3. Understand data augmentation for image tasks.
4. Evaluate with accuracy, per-class F1, confusion matrix.
5. Compare training efficiency of custom CNN vs pretrained model.

## 4 · Problem Statement

Given a 32×32 RGB image, classify it into one of 10 categories.

## 5 · Why This Project Matters

- Image classification is fundamental to computer vision.
- CIFAR-10 teaches CNN basics with manageable compute.
- Transfer learning with modern backbones shows the power of pretraining.
- Understanding small-image classification generalizes to larger tasks.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Train images** | 50,000 |
| **Test images** | 10,000 |
| **Image size** | 32×32 RGB |
| **Classes** | 10 (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck) |
| **Source** | torchvision (auto-download) |

## 7 · Dataset Source and License Notes

- **Source**: CIFAR-10 by Alex Krizhevsky (2009).
- **License**: Public / research use.
- **Citation**: Krizhevsky, A. (2009). Learning Multiple Layers of Features from Tiny Images.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["torch", "torchvision", "timm"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']
NUM_CLASSES = 10
BATCH_SIZE = 128
TEST_BATCH_SIZE = 256
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

CIFAR-10 auto-downloads via torchvision.

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_test)

print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")
print(f"Classes: {CLASSES}")

## 12 · Data Validation Checks

In [ ]:
# Check class distribution using .targets
targets = np.array(train_dataset.targets)
print("Train class distribution:")
for i, name in enumerate(CLASSES):
    count = (targets == i).sum()
    print(f"  {name}: {count}")

print(f"\nImage shape: {train_dataset[0][0].shape}")
print(f"Label range: {targets.min()} to {targets.max()}")

## 13 · Exploratory Data Analysis

In [ ]:
# Sample images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
raw_dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=False)
for i in range(10):
    # Find first image of each class
    idx = (targets == i).nonzero()[0][0]
    img = raw_dataset[idx][0]
    ax = axes[i // 5, i % 5]
    ax.imshow(img)
    ax.set_title(CLASSES[i])
    ax.axis("off")
plt.suptitle("Sample Images — One Per Class", fontsize=14)
plt.tight_layout()
plt.show()

## 14 · Target Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
counts = [int((targets == i).sum()) for i in range(NUM_CLASSES)]
ax.bar(CLASSES, counts, color="steelblue", edgecolor="black")
ax.set_title("Class Distribution (Train)")
ax.set_ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print("CIFAR-10 is perfectly balanced: 5000 images per class.")

## 15 · Train/Validation/Test Split Strategy

CIFAR-10 has a fixed 50K/10K train/test split. We use the official split.

## 16 · Preprocessing Strategy

- **Normalization**: Per-channel mean/std (CIFAR-10 statistics).
- **Augmentation (train)**: RandomHorizontalFlip + RandomCrop(32, padding=4).
- **No augmentation (test)**: Only normalize.

## 17 · Feature Engineering

No manual feature engineering — CNNs learn features automatically from raw pixels.

## 18 · Baseline Model — Simple CNN

A 3-layer CNN with batch norm and dropout.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2), nn.Dropout(0.25),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.5),
            nn.Linear(128, NUM_CLASSES),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = SimpleCNN().to(DEVICE)
print(f"CNN parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Train CNN
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=TEST_BATCH_SIZE, shuffle=False, num_workers=0)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

print("Training Simple CNN (3 epochs)...")
for epoch in range(3):
    model.train()
    running_loss = 0
    correct = 0
    total = 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    print(f"  Epoch {epoch+1}: loss={running_loss/len(train_loader):.4f}, acc={100*correct/total:.1f}%")

# Evaluate CNN
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
cnn_acc = accuracy_score(all_labels, all_preds)
cnn_f1 = f1_score(all_labels, all_preds, average="weighted")
print(f"\nCNN Test Accuracy: {cnn_acc:.4f}, F1: {cnn_f1:.4f}")

## 19 · LazyPredict Benchmark

**Not applicable** for image classification — LazyPredict works on tabular data only.

For CV projects, we compare architectures directly instead.

## 20 · FLAML AutoML Run

**Not applicable** for image classification — FLAML is designed for tabular tasks.

Instead, we compare the CNN baseline against a pretrained ConvNeXt V2 backbone.

## 21 · Additional Model: ConvNeXt V2 Atto (Transfer Learning)

ConvNeXt V2 Atto is the smallest variant (~3.7M params), suitable for fine-tuning on small images.

In [ ]:
# Free CNN memory
del model, optimizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ConvNeXt V2 Atto via timm
import timm

convnext = timm.create_model("convnextv2_atto", pretrained=True, num_classes=NUM_CLASSES)
convnext = convnext.to(DEVICE)
print(f"ConvNeXt V2 Atto parameters: {sum(p.numel() for p in convnext.parameters()):,}")

# Use a small subset for speed (3000 train, 1000 test)
subset_train_idx = np.random.RandomState(SEED).choice(len(train_dataset), 3000, replace=False)
subset_test_idx = np.random.RandomState(SEED).choice(len(test_dataset), 1000, replace=False)
sub_train = Subset(train_dataset, subset_train_idx)
sub_test = Subset(test_dataset, subset_test_idx)

sub_train_loader = DataLoader(sub_train, batch_size=64, shuffle=True, num_workers=0)
sub_test_loader = DataLoader(sub_test, batch_size=TEST_BATCH_SIZE, shuffle=False, num_workers=0)

optimizer2 = optim.AdamW(convnext.parameters(), lr=1e-4, weight_decay=0.01)
criterion2 = nn.CrossEntropyLoss()

print("Fine-tuning ConvNeXt V2 Atto (2 epochs on 3K subset)...")
for epoch in range(2):
    convnext.train()
    running_loss = 0
    correct = 0
    total = 0
    for images, labels in sub_train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer2.zero_grad()
        outputs = convnext(images)
        loss = criterion2(outputs, labels)
        loss.backward()
        optimizer2.step()
        running_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    print(f"  Epoch {epoch+1}: loss={running_loss/len(sub_train_loader):.4f}, acc={100*correct/total:.1f}%")

# Evaluate ConvNeXt on subset test
convnext.eval()
cx_preds, cx_labels = [], []
with torch.no_grad():
    for images, labels in sub_test_loader:
        images = images.to(DEVICE)
        outputs = convnext(images)
        cx_preds.extend(outputs.argmax(1).cpu().numpy())
        cx_labels.extend(labels.numpy())
cx_preds = np.array(cx_preds)
cx_labels = np.array(cx_labels)
cx_acc = accuracy_score(cx_labels, cx_preds)
cx_f1 = f1_score(cx_labels, cx_preds, average="weighted")
print(f"\nConvNeXt V2 Atto Test Accuracy: {cx_acc:.4f}, F1: {cx_f1:.4f}")

## 22 · Top 3 Model Selection

For this CV project, our models are:
1. Simple CNN (full dataset)
2. ConvNeXt V2 Atto (subset fine-tuned)

In [ ]:
print("Model Comparison:")
print(f"  Simple CNN    — Acc: {cnn_acc:.4f}, F1: {cnn_f1:.4f} (50K train, 3 epochs)")
print(f"  ConvNeXt V2   — Acc: {cx_acc:.4f}, F1: {cx_f1:.4f} (3K train, 2 epochs)")

if cnn_acc > cx_acc:
    print("\nSimple CNN wins on accuracy (trained on full dataset).")
else:
    print("\nConvNeXt V2 wins despite training on only 3K samples — transfer learning is powerful!")

## 23 · Final Training and Evaluation

In [ ]:
# Full evaluation of best model (CNN on full test set)
print("CNN — Full Classification Report:")
print(classification_report(all_labels, all_preds, target_names=CLASSES))

fig, ax = plt.subplots(figsize=(10, 8))
from sklearn.metrics import ConfusionMatrixDisplay
cm = confusion_matrix(all_labels, all_preds)
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, colorbar=False, xticks_rotation=45)
ax.set_title(f"CNN Confusion Matrix (Acc={cnn_acc:.4f})")
plt.tight_layout()
plt.show()

## 24 · Error Analysis

In [ ]:
# Most confused class pairs
misclassed = (all_labels != all_preds)
n_wrong = misclassed.sum()
print(f"Misclassified: {n_wrong} / {len(all_labels)} ({100*n_wrong/len(all_labels):.1f}%)")

# Per-class accuracy
print("\nPer-class accuracy:")
for i, name in enumerate(CLASSES):
    mask = all_labels == i
    cls_acc = (all_preds[mask] == i).mean()
    print(f"  {name:12s}: {cls_acc:.4f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Cat vs Dog** is the hardest pair — both are small furry animals at 32×32.
- **Ship and Truck** are easy — distinct shapes.
- **Transfer learning** (ConvNeXt V2) achieves competitive results with far less data.

**Practical takeaway:** For production image classification, always start with a pretrained backbone.

## 26 · Limitations

1. CIFAR-10 is 32×32 — far smaller than real-world images.
2. Only 10 classes — real tasks have hundreds/thousands.
3. ConvNeXt trained on subset (3K) — full fine-tuning would perform better.
4. No test-time augmentation or ensemble.
5. No hyperparameter tuning.

## 27 · How to Improve This Project

1. Train ConvNeXt V2 on the full 50K dataset.
2. Use learning rate schedulers (cosine annealing).
3. Apply CutMix / MixUp augmentation.
4. Try DINOv3 backbone for self-supervised features.
5. Use test-time augmentation for better accuracy.

## 28 · Production Considerations

- Use ONNX export for deployment.
- Batch inference for throughput.
- Monitor data drift with distribution checks.
- Consider model distillation for edge deployment.

## 29 · Common Mistakes

1. Not normalizing images per-channel.
2. Using too high a learning rate for fine-tuning pretrained models.
3. Training for too many epochs on small datasets (overfitting).
4. Not using data augmentation for small training sets.
5. Evaluating ConvNeXt on train data instead of held-out test.

## 30 · Mini Challenge / Exercises

1. Train CNN for 10 epochs — how much does accuracy improve?
2. Try ResNet-18 from torchvision — compare with ConvNeXt.
3. Remove data augmentation — how much does accuracy drop?
4. Visualize learned filters from the first conv layer.
5. Try a simple MLP on flattened pixels — how bad is it?

## 31 · Final Summary / Key Takeaways

1. **CNNs** learn hierarchical features from raw pixels.
2. **Transfer learning** outperforms training from scratch, especially with limited data.
3. **Data augmentation** is essential for image classification.
4. **CIFAR-10** is a great learning benchmark but far simpler than real-world tasks.
5. **ConvNeXt V2 Atto** is the smallest modern backbone — great for constrained environments.

## Save Metrics

In [ ]:
metrics_out = {
    "Simple_CNN": {"accuracy": round(float(cnn_acc), 4), "f1": round(float(cnn_f1), 4)},
    "ConvNeXt_V2_Atto": {"accuracy": round(float(cx_acc), 4), "f1": round(float(cx_f1), 4)},
}
metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))